In [ ]:
#]'!nvidia-smi
!nvidia-smi

Wed Dec 24 00:53:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.utils import Sequence
from random import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import math
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPool2D , Flatten, BatchNormalization, MaxPooling2D, Activation, Dropout
from keras.preprocessing.image import ImageDataGenerator
import numpy as np

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip '/content/drive/My Drive/archive (1).zip' -d '/content/input'

**Get dataset**

In [ ]:
#!pip install opendatasets
#import opendatasets as od
#od.download('https://www.kaggle.com/kmader/satellite-images-of-hurricane-damage')

**Build database with filenames and labels**

In [ ]:
def get_files(directory):
    df = pd.DataFrame(columns=['file','target'])
    damage = glob.glob(directory+'/damage/*.jpeg')
    no_damage = glob.glob(directory+'/no_damage/*jpeg')
    for f in damage:
        df = df.append({'file':f,'target':'1'},ignore_index=True)
    for f in no_damage:
        df = df.append({'file':f,'target':'0'},ignore_index=True)
    return df

def get_image(file_path):
    return mpimg.imread(file_path)/255

In [ ]:
df_train = pd.concat([get_files('input/train_another'),
                      get_files('input/test_another'),
                      get_files('input/validation_another')],axis=0)
df_test = get_files('input/test')

In [ ]:
df_train.reset_index(inplace=True,drop=True)

In [ ]:
df_train.head()

In [ ]:
df_train.shape

**Visualize some data**

In [ ]:
#ax = sns.histplot(x='target', data=df_train, bins=2)
#ax.set_xticks([0.25,0.75])
#ax.set_xticklabels([0,1])
#plt.show()

In [ ]:
img = get_image(df_train[df_train['target']=='1']['file'].iloc[0])
img.shape

fig = plt.figure(figsize=(5,5))
plt.imshow(img)

In [ ]:
fig = plt.figure(figsize=(5,5))
plt.imshow(img)

In [ ]:
fig = plt.figure(figsize=(5,5))
plt.imshow(get_image(df_train[df_train['target']=='0']['file'].iloc[0]))

**Define the data structure for the model**

In [ ]:
train_gen = ImageDataGenerator(validation_split=0.1)
test_gen = ImageDataGenerator()

train_data = train_gen.flow_from_dataframe(
    dataframe = df_train,
    x_col = 'file',
    y_col = 'target',
    target_size = (256,256),
    color_mode = 'rgb',
    class_mode = 'binary',
    shuffle = True,
    subset = 'training',
    batch_size=100
)

val_data = train_gen.flow_from_dataframe(
    dataframe = df_train,
    x_col = 'file',
    y_col = 'target',
    target_size = (256,256),
    color_mode = 'rgb',
    class_mode = 'binary',
    shuffle = False,
    subset = 'validation',
    batch_size=100
)

test_data = test_gen.flow_from_dataframe(
    dataframe = df_test,
    x_col = 'file',
    y_col = 'target',
    target_size = (256,256),
    color_mode = 'rgb',
    class_mode = 'binary',
    shuffle = False,
    batch_size=100
)

**Create Custom Model 2**

In [ ]:
model2 = keras.models.Sequential([
    keras.layers.Conv2D(filters=96, kernel_size=(11,11), strides=(4,4), activation='relu', input_shape=(256,256,3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(pool_size=(3,3), strides=(2,2)),
    keras.layers.Conv2D(filters=256, kernel_size=(5,5), strides=(1,1), activation='relu', padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(pool_size=(3,3), strides=(2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation='sigmoid')
])
model2.compile(
    optimizer=tf.optimizers.Adam(lr=0.000001),
    loss='binary_crossentropy',
    metrics=['accuracy','Recall']
)

model2.summary()

**Fit model**

In [ ]:
history = model2.fit(train_data,epochs=9,validation_data=val_data)

**Evaluate model**

In [ ]:
#plotting the Accuracy of test and training sets
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#plotting the loss of test and training sets
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
y_pred = model2.predict(test_data)
y_pred = np.where(y_pred > 0.5,1,0)

In [ ]:
print(classification_report(test_data.labels,y_pred))

In [ ]:
con_mat_df = pd.DataFrame(confusion_matrix(test_data.labels,y_pred))

In [ ]:
fpr_model2,tpr_model2,threshold_model2=roc_curve(test_data.labels,y_pred)
plt.plot(fpr_model2,tpr_model2)
plt.ylabel('tpr')
plt.xlabel('fpr')
plt.show()

In [ ]:
auc_model2=roc_auc_score(test_data.labels,y_pred)
print("AUC FOR MODEL 2 %f"%auc_model2)

**Testing On VGG 16**

In [ ]:
model = Sequential()

model.add(Conv2D(input_shape=(256,256,3),filters=64,kernel_size=(11,11),strides=(4,4),padding="same", activation="relu"))
model.add(Conv2D(filters=64,kernel_size=(3,3),padding="same", activation="relu"))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))

model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))

model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))

model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
model.add(keras.layers.BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))


model.add(Flatten())
model.add(Dense(units=4096,activation="relu"))
model.add(Dense(units=4096,activation="relu"))
model.add(Dense(units=1, activation="sigmoid"))

In [ ]:
model.compile(
    optimizer=tf.optimizers.Adam(lr=0.000001),
    loss='binary_crossentropy',
    metrics=['accuracy','Recall'])
model.summary()

In [ ]:
history = model.fit(train_data,epochs=3,validation_data=val_data)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#plotting the loss of test and training sets
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
y_pred = model.predict(test_data)
y_pred = np.where(y_pred > 0.5,1,0)

In [ ]:
print(classification_report(test_data.labels,y_pred))

In [ ]:
con_mat_df = pd.DataFrame(confusion_matrix(test_data.labels,y_pred))

In [ ]:
figure = plt.figure(figsize=(12, 6))
sns.heatmap(con_mat_df, annot=True,fmt='d')
plt.tight_layout()
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
fpr_model,tpr_model,threshold_model=roc_curve(test_data.labels,y_pred)
plt.plot(fpr_model,tpr_model)
plt.ylabel('tpr')
plt.xlabel('fpr')
plt.show()

In [ ]:
auc_model=roc_auc_score(test_data.labels,y_pred)
print("AUC FOR VGG 16 %f"%auc_model)

**Trying on AlexNet**

In [ ]:
AlexNet = Sequential()


AlexNet.add(Conv2D(filters=96, input_shape=(256,256,3), kernel_size=(11,11), strides=(4,4), padding='same'))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))
AlexNet.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'))

AlexNet.add(Conv2D(filters=256, kernel_size=(5, 5), strides=(1,1), padding='same'))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))
AlexNet.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'))

AlexNet.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='same'))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))

AlexNet.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='same'))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))

AlexNet.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), padding='same'))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))
AlexNet.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same'))

#Passing it to a Fully Connected layer
AlexNet.add(Flatten())

AlexNet.add(Dense(4096, input_shape=(32,32,3,)))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))

AlexNet.add(Dropout(0.4))

#2nd Fully Connected Layer
AlexNet.add(Dense(4096))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))

AlexNet.add(Dropout(0.4))

#3rd Fully Connected Layer
AlexNet.add(Dense(1000))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('relu'))

AlexNet.add(Dropout(0.4))

#Output Layer
AlexNet.add(Dense(1))
AlexNet.add(BatchNormalization())
AlexNet.add(Activation('sigmoid'))

#Model Summary
AlexNet.summary()

In [ ]:
AlexNet.compile(
    optimizer=tf.optimizers.Adam(lr=0.000001),
    loss='binary_crossentropy',
    metrics=['accuracy','Recall'])

In [ ]:
history = AlexNet.fit(train_data,epochs=5,validation_data=val_data)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#plotting the loss of test and training sets
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
y_pred = AlexNet.predict(test_data)
y_pred = np.where(y_pred > 0.5,1,0)

In [ ]:
print(classification_report(test_data.labels,y_pred))

In [ ]:
fpr_AlexNet,tpr_AlexNet,threshold_AlexNet=roc_curve(test_data.labels,y_pred)
plt.plot(fpr_AlexNet,tpr_AlexNet)
plt.ylabel('tpr')
plt.xlabel('fpr')
plt.show()

In [ ]:
auc_AlexNet=roc_auc_score(test_data.labels,y_pred)
print("AUC FOR AlexNet %f"%auc_AlexNet)

**Create custom model 1**

In [ ]:
train_idx =  df_train['file'].values
y = df_train['target'].values
test_idx =  df_test['file'].values
y_test = df_test['target'].values
x_train,x_valid,y_train,y_valid = train_test_split(train_idx,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
x_train_img = np.zeros((len(x_train),128*128*3))
for i,f in enumerate(x_train):
    x_train_img[i,:] = get_image(f).reshape(-1)

x_train_img.shape

In [ ]:
x_valid_img = np.zeros((len(x_valid),128*128*3))
for i,f in enumerate(x_valid):
    x_valid_img[i,:] = get_image(f).reshape(-1)

x_valid_img.shape

In [ ]:
x_test_img = np.zeros((len(test_idx),128*128*3))
for i,f in enumerate(test_idx):
    x_test_img[i,:] = get_image(f).reshape(-1)

x_test_img.shape

In [ ]:
model = SVC()
model.fit(x_train_img,y_train)

In [ ]:
y_pred = model.predict(x_test_img)
print(classification_report(y_test,y_pred))

In [ ]:
con_mat_df = pd.DataFrame(confusion_matrix(y_test,y_pred))

In [ ]:
figure = plt.figure(figsize=(12, 6))
sns.heatmap(con_mat_df, annot=True,fmt='d')
plt.tight_layout()
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
Res = tf.keras.applications.ResNet50V2(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling='max',
)

In [ ]:
Dis_model = keras.models.Sequential([
    keras.layers.Conv2D(filters=96, kernel_size=(11,11), strides=(1,1), activation='relu', input_shape=(256,256,3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(pool_size=(3,3), strides=(2,2)),
    keras.layers.Conv2D(filters=256, kernel_size=(5,5), strides=(1,1), activation='relu', padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(pool_size=(3,3), strides=(2,2)),
    keras.layers.Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), activation='relu', padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), activation='relu', padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(filters=3, kernel_size=(3,3), strides=(1,1), activation='relu', padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPool2D(pool_size=(3,3), strides=(1,1)),
    keras.layers.Flatten(),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation='sigmoid')
])
Dis_model.compile(
    optimizer=tf.optimizers.Adam(lr=0.000001),
    loss='binary_crossentropy',
    metrics=['accuracy','Recall']
)

Dis_model.summary()

In [ ]:
history = Dis_model.fit(train_data,epochs=6,validation_data=val_data)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#plotting the loss of test and training sets
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
y_pred = Dis_model.predict(test_data)
y_pred = np.where(y_pred > 0.5,1,0)

In [ ]:
print(classification_report(test_data.labels,y_pred))

In [ ]:
con_mat_df = pd.DataFrame(confusion_matrix(test_data.labels,y_pred))

In [ ]:
figure = plt.figure(figsize=(12, 6))
sns.heatmap(con_mat_df, annot=True,fmt='d')
plt.tight_layout()
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
fpr_Dis_model,tpr_Dis_model,threshold_Dis_model=roc_curve(test_data.labels,y_pred)
plt.plot(fpr_Dis_model,tpr_Dis_model)
plt.ylabel('tpr')
plt.xlabel('fpr')
plt.show()

In [ ]:
auc_Dis_model=roc_auc_score(test_data.labels,y_pred)
print("AUC FOR CUSTOM MODEL 1 %f"%auc_Dis_model)

**RESULT**

In [ ]:
import numpy as np

mapper = dict()
mapper[0] = 'Not Damaged'
mapper[1] = 'Damaged'

PATHS = ['/content/input/test/damage/-93.578271_30.779923999999998.jpeg','/content/input/test/no_damage/-95.062203_29.82817.jpeg']


from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input


my_image = load_img(PATHS[0], target_size=(256, 256))

img = get_image(PATHS[0])


fig = plt.figure(figsize=(5,5))
plt.imshow(img)


my_image = img_to_array(my_image)
my_image = my_image.reshape((1, my_image.shape[0], my_image.shape[1], my_image.shape[2]))
#my_image = preprocess_input(my_image)


prediction = Dis_model.predict(my_image)


print('Damage caused confidence = ', prediction[0][0]*100, '%')
final_preds = int(np.round(prediction[0][0]))
print("Model's Verdict = ",mapper[final_preds])

In [ ]:
# Prediction Cell
import numpy as np

mapper = dict()
mapper[0] = 'Not Damaged'
mapper[1] = 'Damaged'

PATHS = ['/content/input/test/damage/-93.578271_30.779923999999998.jpeg','/content/input/test/no_damage/-95.062203_29.82817.jpeg']


from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

my_image = load_img(PATHS[1], target_size=(256, 256))

img = get_image(PATHS[1])


fig = plt.figure(figsize=(5,5))
plt.imshow(img)


my_image = img_to_array(my_image)
my_image = my_image.reshape((1, my_image.shape[0], my_image.shape[1], my_image.shape[2]))

prediction = Dis_model.predict(my_image)


print('Damage caused confidence = ', prediction[0][0]*100, '%')
final_preds = int(np.round(prediction[0][0]))
print("Model's Verdict = ",mapper[final_preds])
